# Annual-meeting put IV mechanism test

Separate experiment: current CSV sample and expanded Massive history. **Conclusion: inconclusive.** This tests entry pricing, not protective-put profitability. Choices were frozen before IV differences; historical candidate selection followed related P&L discovery. Original starter notebook and harness are untouched.


In [1]:
import json
import pandas as pd
import annual_meeting_iv_test as study
import annual_meeting_iv_analysis as analysis
from verify_annual_meeting_iv import run as verify
# Set True to retrieve/rebuild using the existing local Massive key; never print it.
REBUILD = False
study.freeze()
display(json.loads((study.OUT / "registration.json").read_text()))


{'id': 'AM-IV-v1', 'hypothesis': 'Annual-meeting entry put IV is lower than matched ordinary-day IV conditional on maturity, actual moneyness and pre-entry realized volatility', 'windows': {'current_csv': ['2024-01-01', '2025-12-31'], 'expanded_early': ['2022-03-07', '2023-12-31'], 'replication_2026': ['2026-01-01', '2026-08-31']}, 'tag': 'annual_meeting_results', 'universe': 'unchanged starter TOP_100; survivorship limitation', 'entry': 'following trading close strictly after filing date; entry-only study has no holding horizon', 'selection': 'starter expiry and paired-strike algorithms, prior-session stock/chain, 3-6m bucket, 5% OTM put; exclude entry moneyness outside 0.90..0.98', 'controls': 'same issuer/year/quarter/weekday, within 63 sessions, no ANY disclosure within 30 calendar days, DTE within 7 trading sessions, actual K/S within .01, pre-entry RV20 ratio within .80..1.25; nearest three', 'primary': 'event midpoint American IV minus mean matched ordinary midpoint American IV,

## Sample counts before IV

Preserve the 2024–2025 CSV event sample; retrieve March 7, 2022–December 2023 from Massive. January–August 2026 is a separate replication feasibility check. Controls use the full Massive all-tag calendar, not CSV absence.


In [2]:
if REBUILD:
    study.counts()
display(pd.read_csv(study.OUT / "calendar_counts.csv"))


,window,events,calendar_matched_upper_bound,count_gate
0,current_csv,150,73,True
1,expanded_early,140,64,True
2,replication_2026,69,36,False


## Entry inputs

Starter 3–6-month expiry and 5% OTM strike algorithms use the prior session. Entry is the first trading close strictly after the filing date. Valid latest bid/ask quotes must precede 16:00 ET and be no older than 60 seconds. RV uses adjusted-close returns ending strictly before entry. Actual entry K/S is matched, as are expiry and RV. No exit prices are needed.


In [3]:
if REBUILD:
    study.collect()
for window in ["current_csv", "expanded_early"]:
    completion = json.loads((study.OUT / window / "collection_complete.json").read_text())
    display({"window": window, **completion})


{'window': 'current_csv', 'entries': 361, 'iv_calculated': False}

{'window': 'expanded_early', 'entries': 336, 'iv_calculated': False}

## American put IV and conditional comparison

Invert a 200-step American CRR tree at the quote midpoint, with trailing paid dividends as a continuous yield proxy and the starter constant rate. Regress each event-minus-control IV difference on differences in pre-entry RV, log maturity and actual K/S. Intercept estimates are descriptive with these small samples. Dependence links repeated companies and overlapping volatility-input/control windows. Confidence intervals require 40 events and five dependence components; the 120-comparison discovery correction remains.


In [4]:
if REBUILD:
    analysis.run()
results = pd.read_csv(study.OUT / "results.csv")
primary = results[(results.variant == "primary") & (results.quote_age == 60)].copy()
for col in ["mean_event_iv", "mean_ordinary_iv", "mean_iv_difference", "adjusted_iv_difference"]:
    primary[col] *= 100
display(primary[["window", "n", "companies", "dependence_groups", "mean_event_iv", "mean_ordinary_iv", "mean_iv_difference", "adjusted_iv_difference", "conclusion"]])
print("IV columns are annualized percentage points, not percent changes or P&L.")


IV columns are annualized percentage points, not percent changes or P&L.


,window,n,companies,dependence_groups,mean_event_iv,mean_ordinary_iv,mean_iv_difference,adjusted_iv_difference,conclusion
0,current_csv,9,8.0,2.0,24.575582,26.578096,-2.002514,-2.503296,inconclusive
9,expanded_early,5,5.0,3.0,31.355531,30.132858,1.222673,-4.438441,inconclusive


In [5]:
display(pd.read_csv(study.OUT / "exclusion_counts.csv"))
sensitivity = results[results.quote_age == 60].copy()
sensitivity["adjusted_iv_difference"] *= 100
display(sensitivity[["window", "variant", "n", "adjusted_iv_difference", "conclusion"]])


,window,quote_age,reason,count
0,current_csv,60,no_control_after_calendar,64
1,current_csv,60,no_control_after_maturity,31
2,current_csv,60,no_control_after_moneyness,16
3,current_csv,60,unusable_quote,12
4,current_csv,60,no_control_after_volatility,8
5,current_csv,60,entry_moneyness,6
6,current_csv,60,no_control_after_quote,3
7,current_csv,60,no_expiry,1
8,current_csv,300,no_control_after_calendar,71
9,current_csv,300,no_control_after_maturity,33


,window,variant,n,adjusted_iv_difference,conclusion
0,current_csv,primary,9,-2.503296,inconclusive
1,current_csv,bid,9,-2.681932,inconclusive
2,current_csv,ask,9,-2.326656,inconclusive
3,current_csv,rv60,9,-1.980999,inconclusive
4,current_csv,rate0,9,-2.452950,inconclusive
5,current_csv,rate6,9,-2.537445,inconclusive
6,current_csv,dividend0,9,-2.521343,inconclusive
7,current_csv,tree400,9,-2.515687,inconclusive
9,expanded_early,primary,5,-4.438441,inconclusive
10,expanded_early,bid,5,-4.403398,inconclusive


In [6]:
verify()


1655 sessions in the calendar; last completed session 2026-10-02. Sessions per year: 2022: 251, 2023: 250, 2024: 252, 2025: 250, 2026: 251
          window   variant  quote_age  n  mean_iv_difference  adjusted_iv_difference  dependence_groups
     current_csv   primary       60.0  9           -0.020025               -0.025033                2.0
     current_csv       bid       60.0  9           -0.020832               -0.026819                2.0
     current_csv       ask       60.0  9           -0.019245               -0.023267                2.0
     current_csv      rv60       60.0  9           -0.020025               -0.019810                2.0
     current_csv     rate0       60.0  9           -0.019569               -0.024529                2.0
     current_csv     rate6       60.0  9           -0.020294               -0.025374                2.0
     current_csv dividend0       60.0  9           -0.020346               -0.025213                2.0
     current_csv   tree400   

## Interpretation

Current sample: 9 matched events, raw IV difference −2.00 points and adjusted difference −2.50 points. Expanded early sample: 5 matched events, raw difference +1.22 points and adjusted difference −4.44 points. With 60-return RV adjustment, the expanded estimate reverses to +19.44 points; four regression coefficients on five events are unstable. Neither sample passes count/dependence gates, so no valid registered interval is available. The 2026 calendar upper bound is 36 and it was not priced.

**Inconclusive:** the data do not establish cheaper post-meeting insurance or absence of an effect. Fixed rates, continuous historical dividend proxies, stock closing prices rather than quote-synchronous stock ticks, static starter-universe survivorship, scheduling/earnings confounds, retrospective tags, and small matched samples limit interpretation. Quote bid/ask, rates, dividends, tree steps, and RV sensitivities are reported; they do not establish executable returns. Entry-only IV has no fixed holding horizon: earlier P&L horizon studies remain separate. See `annual_meeting_iv_results/RESULTS.md` and the frozen registration.
